<div style="width:100%; background-color:#181818; color:#f1f1f1; padding:30px 0; text-align:center; border-radius:10px;">

  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExcXVodWNsM3Bia3duZGljZzRqMTI2MGFiZjlkZzBwcmhuaWxydjlpaiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/AOSwwqVjNZlDO/giphy.gif" alt="Beam Oscillation" width="400" style="border-radius:10px;">

  <h3 style="color:#ffffff; margin-top:15px;"><b>Manual Flexural Method. all deformations</b></h3>

  <p><b>Author:</b> <a href="http://caceli.net/" style="color:#3ea6ff; text-decoration:none;">Msc. Ing. Carlos Andrés Celi Sánchez</a></p>
  <p><b>Course:</b> Matrix Analisys</p>
  <p><b>Year:</b> APRIL - 2026</p>

</div>

#### Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [2]:
from repo_maxtrix_analisys import *

ModuleNotFoundError: No module named 'repo_maxtrix_analisys'

![alt text](image-3.png)

#### Data

In [ ]:
w = 2                       # T/m
L = 4                       # m
vb = 0.30                   # m
vh = 0.30                   # m
E = 2100000                 # T/m2
T1 = 9                      # celcius
T2 = 18                     # celcius
alfa = 9.9e-6               # m/m/celcius
dP = 0.008                  # previus deformation
dR1 = 0.001                 # rad
dR2 = 0.009                 # m
dR3 = 0.012                 # m

#------- Geometry---------
XF = np.array([0, L/2, 0.99*L, 1.01*L, L + L/2, L + 0.99*L, L + 1.01*L, 2*L + L/2, 3*L])
YF = np.zeros(len(XF))

XV = np.array([0, L/2, 1.00*L, 1.00*L, L + L/2, L + 1.00*L, L + 1.00*L, 2*L + 1.00*L/2, 2*L + 1.00*L/2, 3*L])
YV = np.zeros(len(XV))

#### Calculations

In [ ]:
I = vb*vh**3 / 12

##### Flex Matrix
![alt text](image-4.png)

In [ ]:
F11 = L / (3*E*I)
F21 = -L / (6*E*I)
F31 = 0
F12 = F21
F22 = 2*L / (3*E*I)
F32 = L / (6*E*I)
F13 = F31
F23 = F32
F33 = 2*L / (3*E*I)

F = np.array([[F11, F12, F13],
              [F21, F22, F23],
              [F31, F32, F33]])

F_df = pd.DataFrame(F, columns=['Q1','Q2','Q3'], index = ['Q1','Q2','Q3'])
F_df

##### Displacement vector corresponding to the redundants in the real structure $\{D_Q\}$
![alt text](image-5.png)

In [ ]:
DQ = np.array([[-dR1],
               [0],
               [0]])

DQ_df = pd.DataFrame(DQ, columns=['DQ'], index = ['Q1','Q2','Q3'])
DQ_df

##### Displacement vector corresponding to the redundants due to real loads in the released or statically determinate structure

$$ \{D_{QL}\}=
\begin{bmatrix}
D_{QL1} \\
D_{QL2} \\
\vdots \\
D_{QLn}
\end{bmatrix}
$$

In [ ]:
DQL = np.array([[-w*L**3 / (24*E*I)],
                [w*L**3 / (24*E*I)],
                [w*L**3 / (16*E*I)]])

DQL_df = pd.DataFrame(DQL, columns=['DQL'], index = ['Q1','Q2','Q3'])
DQL_df

##### Displacement vector corresponding to the redundants due to temperature effects in the released or statically determinate structure

$$\{D_{QT}\}=
\begin{bmatrix}
D_{QT1} \\
D_{QT2} \\
\vdots \\
D_{QTn}
\end{bmatrix}
$$

In [ ]:
DQT = L /2 * (alfa*(T1-T2)) / vh * np.array([[-1],
                                             [2],
                                             [2]])

DQT_df = pd.DataFrame(DQT, columns=['DQT'], index = ['Q1','Q2','Q3'])
DQT_df

##### Displacement vector corresponding to the redundants due to initial deformations in the released or statically determinate structure

$$ \{D_{QP}\}\begin{bmatrix}
D_{QP1} \\
D_{QP2} \\
\vdots \\
D_{QPn}
\end{bmatrix}
$$

In [ ]:
DQP = np.array([[0],
                [dP/L],
                [-2*dP/L]])

DQP_df = pd.DataFrame(DQP, columns=['DQP'], index = ['Q1','Q2','Q3'])
DQP_df

##### Displacement vector corresponding to the redundants due to support displacements in the released or statically determinate structure

$$\{D_{QR}\}=
\begin{bmatrix}
D_{QR1} \\
D_{QR2} \\
\vdots \\
D_{QRn}
\end{bmatrix}
$$

In [ ]:
DQR = np.array([[-dR2/L],
                [-2*dR2/L + dR3/L],
                [dR2/L - 2*dR3/L]])

DQR_df = pd.DataFrame(DQR, columns=['DQR'], index = ['Q1','Q2','Q3'])
DQR_df

#### Total displacement vector corresponding to the redundants in the released or statically determinate structure

$$\{D_{QS}\}=
\{D_{QL}\}
+
\{D_{QT}\}
+
\{D_{QP}\}
+
\{D_{QR}\}
$$

In [ ]:
DQS = DQL + DQT + DQP + DQR

DQS_df = pd.DataFrame(DQS, columns=['DQS'], index = ['Q1','Q2','Q3'])
DQS_df

#### Calculation of redundants

$$\{Q\}=[F]^{-1}\left(\{D_Q\}-\{D_{QS}\}\right)
$$

In [ ]:
#------------Due all deformations---------------
Q_all = np.linalg.inv(F) @ (DQ - DQS)
Q_all_df = pd.DataFrame(Q_all, columns=['Q due all deformations'])
#------------only due real loads---------------
Q_L = np.linalg.inv(F) @ (0 - DQL)
Q_L_df = pd.DataFrame(Q_L, columns=['Q due real loads'])
#------------only due climate effects---------------
Q_T = np.linalg.inv(F) @ (0 - DQT)
Q_T_df = pd.DataFrame(Q_T, columns=['Q due climate effects'])
#------------only due previus defformations---------------
Q_P = np.linalg.inv(F) @ (0 - DQP)
Q_P_df = pd.DataFrame(Q_P, columns=['Q due previus defformations'])
#------------only due support diplacement---------------
Q_R = np.linalg.inv(F) @ (0 - DQR)
Q_R_df = pd.DataFrame(Q_R, columns=['Q due support diplacement'])

Q = pd.concat([Q_all_df, Q_L_df, Q_T_df, Q_P_df, Q_R_df], axis = 1, ignore_index= False)
Q

#### Member action vector (flexure) in the real structure

$$\{AM\}=
\{AM_L\}
+
[AM_Q]\{Q\}
$$

Where:

- $\{AM\}$ = member action vector (flexure) in the real structure.
- $\{AM_L\}$ = member action vector (flexure) due to real loads in the released or statically determinate structure.
- $[AM_Q]$ = member action matrix (flexure) due to unit redundants in the released or statically determinate structure.
- $\{Q\}$ = vector of redundants.

![alt text](image-6.png)

##### Calculation of Member action vector (flexure) due to real loads in the released or statically determinate structure $\{AM\}$
![alt text](image-7.png)

In [ ]:
AML = np.array([[0],
                [w*L**(2)/8],
                [0],
                [0],
                [0],
                [0],
                [0],
                [w*L**(2)/4],
                [0],
                ])

##### Calculation of Member action matrix (flexure) due to unit redundants in the released or statically determinate structure. $[AM_Q]$
![alt text](image-8.png)

In [ ]:
AMQ = np.array([[-1, 0, 0],
                [-1/2, 1/2, 0],
                [0, 1, 0],
                [0, 1, 0],
                [0, 1/2, 1/2],
                [0, 0, 1],
                [0, 0, 1],
                [0, 0, 1/2],
                [0, 0, 0],
                ])

##### Calculation of Member action vector (flexure) in the real structure $\{AM\} = \{AM_L\}+[AM_Q]\{Q\}$.

In [ ]:
#------------Due all deformations---------------
AM_all = AML + (AMQ @ Q_all)
AM_all_df = pd.DataFrame(AM_all, columns=['AM due all deformations'])
#------------only due real loads---------------
AM_L = AML + (AMQ @ Q_L)
AM_L_df = pd.DataFrame(AM_L, columns=['AM due real loads'])
#------------only due climate effects---------------
AM_T = AML + (AMQ @ Q_T)
AM_T_df = pd.DataFrame(AM_T, columns=['AM due climate effects'])
#------------only due previus defformations---------------
AM_P = AML + (AMQ @ Q_P)
AM_P_df = pd.DataFrame(AM_P, columns=['AM due previus defformations'])
#------------only due support diplacement---------------
AM_R = AML + (AMQ @ Q_R)
AM_R_df = pd.DataFrame(AM_R, columns=['AM due support displacement'])

AM = pd.concat([AM_all_df, AM_L_df, AM_T_df, AM_P_df, AM_R_df], axis = 1, ignore_index= False)
AM


##### For Plotting - FLEXION

In [ ]:
AM_all = AM_all * -1
AM_L = AM_L * -1
AM_T = AM_T * -1
AM_P = AM_P * -1
AM_R = AM_R * -1

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# --- Datos para los 2 primeros gráficos (all deformations y real loads) ---
datos = [
    (am1, 'blue', 'Flexural Diagram, all defformations'),
    (am2, 'red',  'Flexural Diagram, due Real Loads'),
]

for ax, (am, color, titulo) in zip(axes, datos):
    # Puntos totales
    x_pts = XF
    y_pts = am

    # ---- Primeros 3 puntos: spline cúbico (parabólico) ----
    x_spline = x_pts[:3]
    y_spline = y_pts[:3]
    cs = CubicSpline(x_spline, y_spline, bc_type='natural')
    x_fine = np.linspace(x_spline[0], x_spline[-1], 200)
    y_fine = cs(x_fine)

    # ---- Resto de puntos: líneas rectas ----
    x_resto = x_pts[2:]   # desde el 3er punto en adelante
    y_resto = y_pts[2:]

    # Dibujar spline (primeros 3 puntos)
    ax.plot(x_fine, y_fine, color=color, marker=None, linewidth=1.5)
    ax.fill_between(x_fine, y_fine, 0, alpha=0.3, color=color)

    # Dibujar líneas rectas (resto)
    ax.plot(x_resto, y_resto, color=color, marker=None, linewidth=1.5)
    ax.fill_between(x_resto, y_resto, 0, alpha=0.3, color=color)

    # Marcar todos los puntos con círculos blancos
    ax.plot(x_pts, y_pts, 'o', color='white', markeredgecolor='black', zorder=5)

    # Anotar valores en cada punto
    for xi, yi in zip(x_pts, y_pts):
        ax.annotate(f'{yi:.2f}', (xi, yi), textcoords="offset points",
                    xytext=(0, 8), ha='center', fontsize=8, color=color)

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Distance [m]')
    ax.set_ylabel('Moment [T-m]')
    ax.legend([titulo], loc='upper right')

plt.tight_layout()
plt.show()

#### Member action vector (Shear) in the real structure

$$\{AM\}=
\{AM_L\}
+
[AM_Q]\{Q\}
$$

Where:

- $\{AM\}$ = member action vector (Shear) in the real structure.
- $\{AM_L\}$ = member action vector (Shear) due to real loads in the released or statically determinate structure.
- $[AM_Q]$ = member action matrix (Shear) due to unit redundants in the released or statically determinate structure.
- $\{Q\}$ = vector of redundants.
![alt text](image-9.png)

##### Calculation of Member action vector (Shear) due to real loads in the released or statically determinate structure $\{AM\}$
![alt text](image-10.png)

In [ ]:
AMLv = np.array([[w*L/2],
                 [0],
                 [-w*L/2],
                 [0],
                 [0],
                 [0],
                 [w*L/2],
                 [w*L/2],
                 [-w*L/2],
                 [-w*L/2]])

##### Calculation of Member action matrix (Shear) due to unit redundants in the released or statically determinate structure. $[AM_Q]$
![alt text](image-11.png)

In [ ]:
AMQv = np.array([[1/L, 1/L, 0],
                 [1/L, 1/L, 0],
                 [1/L, 1/L, 0],
                 [0, -1/L, 1/L],
                 [0, -1/L, 1/L],
                 [0, -1/L, 1/L],
                 [0, 0, -1/L],
                 [0, 0, -1/L],
                 [0, 0, -1/L],
                 [0, 0, -1/L]])

##### Calculation of Member action vector (Shear) in the real structure $\{AM\} = \{AM_L\}+[AM_Q]\{Q\}$.

In [ ]:
#------------Due all deformations---------------
AMv_all = AMLv + (AMQv @ Q_all)
AMv_all_df = pd.DataFrame(AMv_all, columns=['AMv due all deformations'])
#------------only due real loads---------------
AMv_L = AMLv + (AMQv @ Q_L)
AMv_L_df = pd.DataFrame(AMv_L, columns=['AMv due real loads'])
#------------only due climate effects---------------
AMv_T = AMLv + (AMQv @ Q_T)
AMv_T_df = pd.DataFrame(AMv_T, columns=['AMv due climate effects'])
#------------only due previus defformations---------------
AMv_P = AMLv + (AMQv @ Q_P)
AMv_P_df = pd.DataFrame(AMv_P, columns=['AMv due previus defformations'])
#------------only due support diplacement---------------
AMv_R = AMLv + (AMQv @ Q_R)
AMv_R_df = pd.DataFrame(AMv_R, columns=['AMv due support displacement'])

AMv = pd.concat([AMv_all_df, AMv_L_df, AMv_T_df, AMv_P_df, AMv_R_df], axis = 1, ignore_index= False)
AMv

##### For Plotting - SHEAR

In [ ]:
am1 = AMv_all.ravel()
am2 = AMv_L.ravel()
am3 = AMv_T.ravel()
am4 = AMv_P.ravel()
am5 = AMv_R.ravel()
ylabel = 'Shear [T]'

Plot_Shear = Manual_Flexural_Method(am1 = am1, am2 = am2, am3 = am3, am4 = am4, am5 = am5,
                                    X = XV, Y = YV, ylabel = ylabel, L = L, title = 'Shear')
Plot_Shear.Plot_AM_Manual_Flexural_Method()